In [1]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(".env 파일에 GROQ_API_KEY 가 없습니다.")
print("GROQ_API_KEY 설정 확인 완료", GROQ_API_KEY[:2])


GROQ_API_KEY 설정 확인 완료 gs


#### Groq API로 호출 가능한 모델 목록 확인

- Groq는 OpenAI 호환 API이므로 `openai` SDK에 `base_url`만 Groq 주소로 지정하면 `client.models.list()`로 모델 목록을 조회할 수 있습니다.
- 목록 조회는 토큰을 사용하지 않습니다.
- 아래 `ChatOpenAI`에서 사용할 모델명이 목록에 있는지 확인하는 용도로도 사용할 수 있습니다.

In [2]:
# openai SDK 불러오기 (Groq도 OpenAI 호환 API라서 같은 SDK 사용)
from openai import OpenAI

GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트

# Groq 키와 엔드포인트로 클라이언트 생성 (GROQ_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 11

allam-2-7b                                    SDAIA
canopylabs/orpheus-arabic-saudi               Canopy Labs
canopylabs/orpheus-v1-english                 Canopy Labs
meta-llama/llama-prompt-guard-2-22m           Meta
meta-llama/llama-prompt-guard-2-86m           Meta
openai/gpt-oss-120b                           OpenAI
openai/gpt-oss-20b                            OpenAI
openai/gpt-oss-safeguard-20b                  OpenAI
qwen/qwen3.8-27b                              Alibaba Cloud
whisper-large-v3                              OpenAI
whisper-large-v3-turbo                        OpenAI


In [3]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [4]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [5]:
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-3.5-turbo-0125")

# Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    # model="qwen/qwen3.8-27b",
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
openai/gpt-oss-120b


In [6]:
try:
    response = llm.invoke(prompt_text)
    print(type(response)) # AIMessage
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: ## 파이썬(Python)이란?

파이썬은 **고수준(high‑level), 인터프리터(interpreter) 방식의 범용 프로그래밍 언어**입니다. 1991년 네덜란드의 프로그래머 **귀도 반 로썸(Guido van Rossum)**이 처음 발표했으며, 현재는 전 세계 개발자 커뮤니티와 기업에서 폭넓게 사용되고 있습니다.  

---

## 1. 파이썬의 주요 특징

| 특징 | 설명 |
|------|------|
| **읽기 쉬운 문법** | 코드가 거의 영어 문장처럼 읽히므로 초보자도 빠르게 학습할 수 있습니다. 들여쓰기(Indentation)로 블록을 구분해 가독성이 뛰어납니다. |
| **동적 타이핑** | 변수 선언 시 타입을 명시하지 않아도 되며, 실행 시점에 타입이 결정됩니다. (`x = 10`, `x = "hello"` 가능) |
| **플랫폼 독립성** | Windows, macOS, Linux 등 대부분의 운영체제에서 동일한 코드를 실행할 수 있습니다. |
| **풍부한 표준 라이브러리** | 파일 입출력, 네트워킹, 웹, 데이터베이스, 멀티스레딩 등 다양한 기능을 제공하는 “배터리 포함(batteries‑included)” 철학을 따릅니다. |
| **확장성** | C, C++, Java 등으로 작성된 모듈을 손쉽게 호출할 수 있어 성능이 중요한 부분은 네이티브 코드로 보완 가능합니다. |
| **인터프리터 기반** | 코드를 컴파일 없이 바로 실행할 수 있어 빠른 프로토타이핑이 가능합니다. (`python script.py`) |
| **멀티패러다임** | 절차적, 객체지향, 함수형 프로그래밍을 모두 지원합니다. |
| **활발한 커뮤니티** | PyPI(Python Package Index)에 30만 개가 넘는 서드파티 패키지가 존재하고, Stack Overflow, GitHub, Reddit 등에서 활발히 질문·답변이 이뤄집니다. |

---

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [8]:
# LCEL
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002B76F0CD280>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002B76F227DD0>, root_client=<openai.OpenAI object at 0x000002B76F0B38C0>, root_async_client=<openai.AsyncOpenAI object at 0x000002B76F226B10>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, ope

In [9]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [10]:
print(type(response)) #str
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
## LangChain이란?

**LangChain**은 **대규모 언어 모델(LLM)**을 활용한 **애플리케이션**을 보다 쉽고 효율적으로 만들 수 있게 도와주는 **프레임워크**입니다.  
LLM을 단순히 “텍스트를 생성하는 도구”로 쓰는 것이 아니라, **다양한 외부 데이터·도구·워크플로우와 결합**해 실제 비즈니스 로직을 수행하도록 설계되었습니다.

---

## 핵심 개념

| 개념 | 설명 | 예시 |
|------|------|------|
| **Chains** | 여러 LLM 호출·프롬프트·처리 단계를 순차·조건부·반복적으로 연결한 흐름 | 질문 → 검색 → 요약 → 답변 생성 |
| **Agents** | 목표를 달성하기 위해 **도구(tool)** 를 동적으로 선택·실행하는 자동화된 의사결정 주체 | “날씨 알려줘” → 검색 API 호출 → 결과 반환 |
| **Memory** | 대화·작업 중 **컨텍스트(이전 대화, 상태)** 를 저장·활용 | 챗봇이 “지난번에 얘기한 여행 일정”을 기억 |
| **Prompt Templates** | 재사용 가능한 프롬프트 형식. 변수 삽입으로 동적 프롬프트 생성 | `"{question}에 대한 답을 3줄 이내로 요약해줘"` |
| **Data Loaders & Retrievers** | 외부 데이터(문서, DB, 웹 등)를 **로드·인덱싱·검색**하는 컴포넌트 | PDF → 텍스트 추출 → 벡터 DB에 저장 → 유사도 검색 |
| **Integrations** | 다양한 외부 서비스와 연동 (SQL, Elasticsearch, Pinecone, OpenAI, Azure, HuggingFace 등) | `SQLToolkit`을 이용해 데이터베이스 질의 자동화 |
| **Callbacks** | 실행 흐름에 대한 **로그·모니터링·디버깅**을 위한 훅 | 토큰 사용량, 응답 시간, 오류 등을 실시간 기록 |
